In [1]:
# --- project bootstrap (added during repo consolidation) ---
# Resolves the project root so the notebook runs from notebooks/ or from the repo root,
# and puts src/ on sys.path so `import laptop_price` works.
import os, sys, pathlib

_here = pathlib.Path.cwd()
for _cand in (_here, *_here.parents):
    if (_cand / "pyproject.toml").exists():
        PROJECT_ROOT = _cand
        break
else:
    raise RuntimeError("pyproject.toml not found - run this notebook inside the repo")

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"project root: {PROJECT_ROOT}")


project root: /app


# Model-Ready Dataset Creation

This notebook transforms preprocessed laptop data into a **model-ready format** for machine learning. The pipeline covers:

- **Feature Encoding**: Converting categorical variables (CPU, GPU, model names) to numerical representations
- **RAM & Storage Processing**: Extracting DDR generation from RAM types, encoding storage configurations
- **Condition (Etat) Encoding**: Ordinal encoding based on price correlation with inference for missing values
- **Feature Selection**: Dropping redundant columns while preserving predictive power
- **Correlation Analysis**: Final heatmap to visualize feature relationships with price


## Libraries

**Imports essential packages for data manipulation and analysis.**
- `pandas` and `numpy` for data processing and numerical operations
- `re` for regular expression pattern matching in text extraction


In [2]:
import numpy as np
import pandas as pd
import csv
import os
import re
from rapidfuzz import process, fuzz
import seaborn as sns
import matplotlib.pyplot as plt

## Data Loading

**Loads the preprocessed laptop dataset from the previous notebook.**
- Source file: `laptops_after_preprocessing.csv` containing cleaned laptop specifications
- Dataset includes CPU/GPU benchmarks, RAM/storage sizes, screen specs, and pricing data


In [3]:
IN = "data/processed/pre_processed_data.csv"
OUT = "data/processed/clustering_optimized_data.csv"
df = pd.read_csv(IN)

## Encoding and Cleaning

**Selects and prepares columns for the final model-ready dataset.**
- Filters dataset to include only relevant features for price prediction
- Retains key specifications: CPU/GPU metrics, RAM/storage, screen attributes
- Removes temporary or preprocessing-specific columns


In [4]:
#keep only necessary columns
#replace df["price_preview"] with df["price_corrected"]
df["price_preview"] = df["price_corrected"]
df.drop(columns=["price_corrected"], inplace=True)
df = df[
    [ 'price_preview', 'created_at', 'spec_Etat', 'model_name',
       'RAM_SIZE', 'SSD_SIZE', 'HDD_SIZE',
       'SCREEN_SIZE', 'SCREEN_RESOLUTION', 'RAM_TYPE', 'mapped_cpu_name',
       'match_score', 'cores', 'cpu_mark', 'tdp', 'gpu_name',
       'gpu_g3d_mark', 'gpu_g2d_mark', 'gpu_tdp',
       'SCREEN_SIZE_SNAPPED', 'SCREEN_RESOLUTION_STD', 'SCREEN_RESOLUTION_ENC',
         'cpu_generation_normalized'
    ]
]
df.columns


Index(['price_preview', 'created_at', 'spec_Etat', 'model_name', 'RAM_SIZE',
       'SSD_SIZE', 'HDD_SIZE', 'SCREEN_SIZE', 'SCREEN_RESOLUTION', 'RAM_TYPE',
       'mapped_cpu_name', 'match_score', 'cores', 'cpu_mark', 'tdp',
       'gpu_name', 'gpu_g3d_mark', 'gpu_g2d_mark', 'gpu_tdp',
       'SCREEN_SIZE_SNAPPED', 'SCREEN_RESOLUTION_STD', 'SCREEN_RESOLUTION_ENC',
       'cpu_generation_normalized'],
      dtype='object')

### Price Column Update

**Replaces `price_preview` with corrected prices from preprocessing.**
- Uses `price_corrected` values to replace original `price_preview` column
- Drops the redundant `price_corrected` column after transfer
- Selects only the model-relevant columns for further processing


## CPU & GPU Processing

**Handles CPU and GPU data for encoding and feature engineering.**
- CPU names mapped to benchmark scores (`cpu_mark`) and specifications
- GPU names processed similarly with G3D/G2D marks for performance metrics
- TDP (Thermal Design Power) values included as efficiency indicators


In [5]:
#let's print the cpu names of the null gpu_g3d_mark rows
print(df[df["gpu_g3d_mark"].isna()]["mapped_cpu_name"])

17       NaN
21       NaN
41       NaN
45       NaN
53       NaN
        ... 
16379    NaN
16380    NaN
16381    NaN
16386    NaN
16387    NaN
Name: mapped_cpu_name, Length: 661, dtype: object


In [6]:
# Drop rows with missing CPU mapping

df = df.dropna(subset=['mapped_cpu_name'])



# Safe GPU lookup helper to avoid IndexError when GPU is missing in gpus.csv

df_gpus = pd.read_csv("data/raw/gpus.csv")



def safe_gpu_info(gpu_name):

    match = df_gpus[df_gpus['gpu_name'] == gpu_name]

    if match.empty:

        print(f"[warn] GPU '{gpu_name}' not found in gpus.csv; leaving g3d/g2d/tdp as NaN")

        return {"g3d_mark": np.nan, "g2d_mark": np.nan, "tdp(w)": np.nan}

    row = match.iloc[0]

    return {"g3d_mark": row['g3d_mark'], "g2d_mark": row['g2d_mark'], "tdp(w)": row['tdp(w)']}



# Fill Intel GMA 4500MHD (only where CPU matches the old Celeron pattern)

gma_info = safe_gpu_info('Intel GMA 4500MHD')

df.loc[

    (df['mapped_cpu_name'].str.contains('Celeron Dual-Core T3')) & (df['gpu_name'] == 'Intel GMA 4500MHD'),

    ['gpu_g3d_mark', 'gpu_g2d_mark', 'gpu_tdp']

] = [gma_info['g3d_mark'], gma_info['g2d_mark'], gma_info['tdp(w)']]



# Fill VIA Chrome9

via_info = safe_gpu_info('VIA Chrome9')

df.loc[

    (df['gpu_name'] == 'VIA Chrome9'),

    ['gpu_g3d_mark', 'gpu_g2d_mark', 'gpu_tdp']

] = [via_info['g3d_mark'], via_info['g2d_mark'], via_info['tdp(w)']]



# Drop rows with the problematic Atom CPU

df = df[~df['mapped_cpu_name'].str.contains('Intel Atom C2338 @ 1.74GHz')]



print(df['cpu_mark'].isna().sum())

# Re-map cpu_mark just in case

df_cpu = pd.read_csv("data/raw/cpus.csv")

def get_cpu_mark(cpu_name):

    match = df_cpu[df_cpu['name'] == cpu_name]

    if not match.empty:

        return match.iloc[0]['cpumark']

    else:

        return np.nan

df['cpu_mark'] = df['mapped_cpu_name'].apply(get_cpu_mark)



df['gpu_g3d_mark'] = df['gpu_g3d_mark'].str.replace(',', '')

df['gpu_g3d_mark'] = pd.to_numeric(df['gpu_g3d_mark'], errors='coerce')



df['gpu_g2d_mark'] = df['gpu_g2d_mark'].str.replace(',', '')

df['gpu_g2d_mark'] = pd.to_numeric(df['gpu_g2d_mark'], errors='coerce')



df['cpu_mark'] = df['cpu_mark'].str.replace(',', '')

df['cpu_mark'] = pd.to_numeric(df['cpu_mark'], errors='coerce')



# Print diagnostics

print(df[df['gpu_g3d_mark'].isna()][['model_name', 'mapped_cpu_name', 'gpu_name']])

print('###################')

print(df[df['gpu_name'].isna()][['model_name', 'mapped_cpu_name']])

print(df['cpu_mark'].isna().sum())


0


Empty DataFrame
Columns: [model_name, mapped_cpu_name, gpu_name]
Index: []
###################
Empty DataFrame
Columns: [model_name, mapped_cpu_name]
Index: []
0


### GPU Mark Cleaning & Type Conversion

**Cleans and converts GPU/CPU benchmark values to numeric format.**
- Removes comma separators from benchmark strings (e.g., '15,000' → '15000')
- Converts `gpu_g3d_mark`, `gpu_g2d_mark`, and `cpu_mark` to numeric types
- Handles edge cases for missing GPUs (Intel GMA 4500MHD, VIA Chrome9)


In [7]:
df_gpus = pd.read_csv("data/raw/gpus.csv")

### Data Type Verification

**Verifies column data types after encoding transformations.**
- Confirms all benchmark columns are now numeric (float64/int64)
- Ensures proper encoding of screen resolution and size
- Validates CPU generation normalization values


In [8]:

df.dtypes

price_preview                float64
created_at                    object
spec_Etat                     object
model_name                    object
RAM_SIZE                      object
SSD_SIZE                      object
HDD_SIZE                      object
SCREEN_SIZE                  float64
SCREEN_RESOLUTION             object
RAM_TYPE                      object
mapped_cpu_name               object
match_score                  float64
cores                        float64
cpu_mark                       int64
tdp                          float64
gpu_name                      object
gpu_g3d_mark                   int64
gpu_g2d_mark                   int64
gpu_tdp                      float64
SCREEN_SIZE_SNAPPED          float64
SCREEN_RESOLUTION_STD         object
SCREEN_RESOLUTION_ENC        float64
cpu_generation_normalized    float64
dtype: object

## Model Name Processing

**Encodes laptop model families into numerical categories.**
- Extracts brand-specific model series (e.g., ThinkPad, MacBook, Inspiron)
- Converts categorical model names to integer encoding for ML compatibility
- Preserves model hierarchy information for price prediction


In [9]:
#get the unique value of models in data
unique_models = df['model_name'].unique()
print(f"Unique models in data: {len(unique_models)}")
print(unique_models)
# Convert cpu_mark and gpu_g3d_mark to numeric
#let's plot the average cpu and gpu mark per model maybe we can create some families of models as a feature


model_cpu_gpu_marks = df.groupby('model_name').agg({'cpu_mark': 'mean', 'gpu_g3d_mark': 'mean'}).reset_index()
plt.figure(figsize=(12, 8))
sns.scatterplot(data=model_cpu_gpu_marks, x='cpu_mark', y='gpu_g3d_mark', hue='model_name')
plt.title('Average CPU Mark vs GPU 3D Mark per Laptop Model')
plt.xlabel('Average CPU Mark')
plt.ylabel('Average GPU 3D Mark')
plt.legend(title='Laptop Model', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


# Group by model and get average CPU and GPU marks
model_performance = df.groupby('model_name').agg({'cpu_mark': 'mean', 'gpu_g3d_mark': 'mean'}).reset_index()

# Define families based on both CPU and GPU performance tiers
# Tier 0: entry (CPU 0-10k, GPU 0-4k)
# Tier 1: mid (CPU 10k-25k, GPU 4k-18k) 
# Tier 2: high-end (CPU 25k+, GPU 18k+)

def classify_model_tier(cpu_mark, gpu_mark):
    if cpu_mark >= 25000 and gpu_mark >= 18000:
        return 'high-end'
    elif cpu_mark >= 10000 and gpu_mark >= 4000:
        return 'mid'
    else:
        return 'entry'

model_performance['tier'] = model_performance.apply(lambda row: classify_model_tier(row['cpu_mark'], row['gpu_g3d_mark']), axis=1)

families = {
    'entry': model_performance[model_performance['tier'] == 'entry']['model_name'].tolist(),
    'mid': model_performance[model_performance['tier'] == 'mid']['model_name'].tolist(),
    'high-end': model_performance[model_performance['tier'] == 'high-end']['model_name'].tolist()
}
#function that given a model name returns its family
def get_model_family(model_name):
    for family, models in families.items():
        if model_name in models:
            return family
    return 'unknown'
#apply the function to create a new column 'model_family'
df['model_family'] = df['model_name'].apply(get_model_family)

# Ordinal encoding for model families based on performance tiers
def encode_model_family(family):
    """
    Encode model families using ordinal encoding:
    entry = 1 (lowest performance)
    mid = 2 (medium performance)
    high-end = 3 (highest performance)
    unknown = 0 (unmapped/unknown)
    """
    encoding_map = {
        'entry': 1,
        'mid': 2,
        'high-end': 3,
        'unknown': 0
    }
    return encoding_map.get(family, 0)

# Apply ordinal encoding to model_family
df['model_family_encoded'] = df['model_family'].apply(encode_model_family)

# Drop the original string column and rename the encoded one
df = df.drop(columns=['model_family', 'model_name'])
df.rename(columns={'model_family_encoded': 'model_family'}, inplace=True)

print(f"Model family distribution after encoding:")
print(df['model_family'].value_counts().sort_index())
print(df['cpu_mark'].isna().sum())

Unique models in data: 47
['IDEAPAD' 'AERO' 'STEALTH' 'ROG' nan 'XPS' 'THINKPAD' 'LATITUDE' 'SWORD'
 'PAVILION' 'ELITEBOOK' 'VIVOBOOK' 'PROBOOK' 'ASPIRE' 'MACBOOK' 'VECTOR'
 'INSPIRON' 'NITRO' 'OMEN' 'ALIENWARE' 'TUF' 'GALAXY' 'ZBOOK' 'SURFACE'
 'BLADE' 'LEGION' 'MAC' 'ZENBOOK' 'PRECISION' 'YOGA' 'PREDATOR' 'STRIX'
 'IMAC' 'KATANA' 'VICTUS' 'ENVY' 'SPECTRE' 'THINKBOOK' 'TRAVELMATE' 'GF'
 'VOSTRO' 'SWIFT' 'SPIN' 'OPTIPLEX' 'DYNABOOK' 'TRANSFORMER' 'COMPAQ']


Model family distribution after encoding:
model_family
0      502
1    11800
2     3126
3      161
Name: count, dtype: int64
0


### Model Family Classification

**Groups laptop models into performance tiers based on CPU/GPU benchmarks.**
- **Entry Tier**: CPU < 10k, GPU < 4k (budget laptops)
- **Mid Tier**: CPU 10k-25k, GPU 4k-18k (mainstream laptops)
- **High-End Tier**: CPU ≥ 25k, GPU ≥ 18k (premium/gaming laptops)
- Applies ordinal encoding: entry=1, mid=2, high-end=3, unknown=0


## Price

In [10]:
## let's investigate rows with price_preview as NaN
nan_price_rows = df[df['price_preview'].isna()]
print(nan_price_rows[['mapped_cpu_name', 'gpu_name', 'cpu_mark', 'gpu_g3d_mark']])
#we drop them
df = df.dropna(subset=['price_preview'])

                      mapped_cpu_name                gpu_name  cpu_mark  \
15750         Intel Core Ultra 9 185H        GeForce RTX 4080     29315   
15751         Intel Core Ultra 7 258V      Intel Arc Graphics     18945   
15752         Intel Core Ultra 7 155U      Intel Arc Graphics     16169   
15753         Intel Core Ultra 7 155H        GeForce RTX 4060     24673   
15754             Intel Core i7-1260P  Intel UHD Graphics 770     16583   
...                               ...                     ...       ...   
15820  Intel Core i5-1135G7 @ 2.40GHz  Intel Iris Xe Graphics      9399   
15821  Intel Core i5-1135G7 @ 2.40GHz  Intel Iris Xe Graphics      9399   
15822  Intel Core i7-10870H @ 2.20GHz        GeForce GTX 1660     14031   
15823  Intel Core i7-10870H @ 2.20GHz        GeForce RTX 2060     14031   
15824  Intel Core i7-10510U @ 1.80GHz      Intel UHD Graphics      6394   

       gpu_g3d_mark  
15750         34449  
15751          5440  
15752          5440  
15753      

## RAM And Storage

In [11]:
def extract_ram_size_gb(ram_str):
    """Extract RAM size in GB from string like '16GB', '512MB'"""
    if pd.isna(ram_str):
        return np.nan
    ram_str = str(ram_str).upper()
    
    # Handle GB
    if 'GB' in ram_str:
        try:
            return float(ram_str.replace('GB', '').strip())
        except:
            return np.nan
    # Handle MB (convert to GB)
    elif 'MB' in ram_str:
        try:
            return float(ram_str.replace('MB', '').strip()) / 1024
        except:
            return np.nan
    return np.nan

def extract_storage_gb(storage_str):
    """Extract storage size in GB"""
    if pd.isna(storage_str):
        return 0
    storage_str = str(storage_str).upper()
    
    # Handle TB
    if 'TB' in storage_str:
        try:
            return float(storage_str.replace('TB', '').strip()) * 1024
        except:
            return 0
    # Handle GB
    elif 'GB' in storage_str:
        try:
            return float(storage_str.replace('GB', '').strip())
        except:
            return 0
    return 0


df['RAM_SIZE'] = df['RAM_SIZE'].apply(extract_ram_size_gb)
df['SSD_SIZE'] = df['SSD_SIZE'].apply(extract_storage_gb)
df['HDD_SIZE'] = df['HDD_SIZE'].apply(extract_storage_gb)

#SSD_SIZE = Nan -> 0
df['SSD_SIZE'] = df['SSD_SIZE'].fillna(0)
#HDD_SIZE = Nan -> 0
df['HDD_SIZE'] = df['HDD_SIZE'].fillna(0)

ram_type_mapping = {
    # Legacy types
    'SDRAM': 0.5,      # Very old
    'DDR': 1,          # DDR1
    'DDR2': 1.5,       # DDR2
    'DDR3': 2,         # DDR3
    'DDR3L': 2,        # DDR3 Low voltage
    'LPDDR3': 2,       # Low power DDR3
    
    # Current mainstream
    'DDR4': 3,         # DDR4
    'DDR4X': 3.2,      # DDR4X (slightly faster)
    'LPDDR4': 3,       # Low power DDR4
    'LPDDR4X': 3.2,    # Low power DDR4X
    
    # Latest generation
    'DDR5': 4,         # DDR5
    'DDR5X': 4.2,      # DDR5X (faster)
    'LPDDR5': 4,       # Low power DDR5
    'LPDDR5X': 4.2,    # Low power DDR5X (fastest mobile)
    
    
    # Apple Unified Memory (similar to LPDDR5 performance)
    'Unified': 4,      # Apple Unified Memory - high performance
    
    # Generic/Mixed
    'DDR4/DDR5': 3.5,  # Mixed compatibility
    'LPDDR': 2.5,      # Generic LPDDR (assume mid)
}

def encode_ram_type(ram_type_str):
    """Encode RAM type to ordinal value"""
    if pd.isna(ram_type_str):
        return np.nan
    ram_str = str(ram_type_str).strip()
    
    # Direct match first
    if ram_str in ram_type_mapping:
        return ram_type_mapping[ram_str]
    
    # Try uppercase match
    ram_upper = ram_str.upper().replace(' ', '')
    for key, value in ram_type_mapping.items():
        if key.upper() == ram_upper or key.upper() in ram_upper:
            return value
    
    return np.nan  # Unknown type

df['ram_type_encoded'] = df['RAM_TYPE'].apply(encode_ram_type)
#lets print the indices of the rows where ram_size is null
missing_ram_indices = df[df['RAM_SIZE'].isna()].index.tolist()

In [12]:
cpu_ram_mapping = {
    "Intel Core i5-8250U @ 1.60GHz": 8,
    "Apple M3 Max 14 Core": 32,
    "Apple M4 Pro 12 Core": 18,
    "Apple M4 10 Core": 16,
    "Intel Core i7-4712HQ @ 2.30GHz": 8,
    "Intel Core i7-1250U": 16,
    "Intel Core i7-1165G7 @ 2.80GHz": 16,
    "Intel Core i7-1185G7 @ 3.00GHz": 16,
    "Intel Core i5-1135G7 @ 2.40GHz": 16,
    "Intel Core i5-6200U @ 2.30GHz": 8,
    "Intel Pentium N4200 @ 1.10GHz": 4,
    "Intel Core i3-1005G1 @ 1.20GHz": 8,
    "Intel Core i3-370M @ 2.40GHz": 4,
    "Celeron Dual-Core T3000 @ 1.80GHz": 2,
    "Apple M3 8 Core": 24,
    "Intel Core i3-2350M @ 2.30GHz": 4,
}

# Fill missing RAM sizes based on CPU name for the missing indices
for idx in missing_ram_indices:
    cpu_name = df.at[idx, 'mapped_cpu_name'] if idx in df.index else None
    if pd.isna(df.at[idx, 'RAM_SIZE']) and cpu_name in cpu_ram_mapping:
        df.at[idx, 'RAM_SIZE'] = cpu_ram_mapping[cpu_name]
        print(f"Index {idx}: Set RAM size to {cpu_ram_mapping[cpu_name]}GB")

# Check how many nulls remain
print(f"\nRemaining null RAM sizes: {df['RAM_SIZE'].isna().sum()}")
print(f"Total rows with RAM info: {(~df['RAM_SIZE'].isna()).sum()}")

# Verify the mappings worked
print("\nVerification of mapped values:")
for cpu_name, ram_size in cpu_ram_mapping.items():
    matches = df.index[(df['mapped_cpu_name'] == cpu_name) & (df['RAM_SIZE'] == ram_size)].tolist()
    if matches:
        print(f"{cpu_name} -> {ram_size}GB (rows: {matches})")

df.drop(columns=['RAM_TYPE'], inplace=True)
df.rename(columns={'ram_type_encoded': 'RAM_TYPE'}, inplace=True)

Index 27: Set RAM size to 8GB
Index 56: Set RAM size to 32GB
Index 97: Set RAM size to 32GB
Index 112: Set RAM size to 18GB
Index 188: Set RAM size to 18GB
Index 342: Set RAM size to 16GB
Index 4819: Set RAM size to 8GB
Index 5603: Set RAM size to 16GB
Index 6377: Set RAM size to 16GB
Index 6910: Set RAM size to 16GB
Index 10238: Set RAM size to 8GB
Index 12123: Set RAM size to 16GB
Index 12828: Set RAM size to 16GB
Index 14477: Set RAM size to 8GB
Index 14865: Set RAM size to 4GB
Index 15334: Set RAM size to 8GB
Index 15439: Set RAM size to 4GB
Index 15458: Set RAM size to 2GB
Index 15745: Set RAM size to 16GB
Index 16067: Set RAM size to 24GB
Index 16170: Set RAM size to 4GB

Remaining null RAM sizes: 0
Total rows with RAM info: 15517

Verification of mapped values:
Intel Core i5-8250U @ 1.60GHz -> 8GB (rows: [19, 27, 43, 61, 135, 286, 402, 2770, 4754, 5726, 5908, 6139, 6247, 6248, 6919, 7060, 7061, 7364, 7558, 7695, 7782, 7861, 7863, 7864, 8328, 8398, 8495, 8663, 9034, 9035, 9036, 9

## SCREEN

In [13]:
df.dtypes
df.drop(columns=['SCREEN_SIZE', 'SCREEN_RESOLUTION', 'SCREEN_RESOLUTION_STD'], inplace=True)

In [14]:
#fill SCREEN_SIZE_SNAPPED NaN with the median
df['SCREEN_SIZE_SNAPPED'] = df['SCREEN_SIZE_SNAPPED'].fillna(df['SCREEN_SIZE_SNAPPED'].median())
df['SCREEN_RESOLUTION_ENC'] = df['SCREEN_RESOLUTION_ENC'].fillna(df['SCREEN_RESOLUTION_ENC'].mode()[0])

In [15]:
df.isna().sum()


price_preview                   0
created_at                      0
spec_Etat                    6437
RAM_SIZE                        0
SSD_SIZE                        0
HDD_SIZE                        0
mapped_cpu_name                 0
match_score                     0
cores                           0
cpu_mark                        0
tdp                             0
gpu_name                        0
gpu_g3d_mark                    0
gpu_g2d_mark                    0
gpu_tdp                         0
SCREEN_SIZE_SNAPPED             0
SCREEN_RESOLUTION_ENC           0
cpu_generation_normalized     607
model_family                    0
RAM_TYPE                        0
dtype: int64

### Missing Value Analysis

**Checks for remaining missing values after encoding.**
- Identifies columns that still contain NaN values
- Key columns with missing data: RAM_SIZE (20), GPU marks (14), cpu_generation (566)
- These will be handled before model training


## Etat

In [16]:
# Let's examine the spec_Etat feature more thoroughly
print("Original spec_Etat value counts:")
original_etat_counts = df['spec_Etat'].value_counts(dropna=False)
print(original_etat_counts)
print(f"\nPercentage of NaN values: {df['spec_Etat'].isna().sum() / len(df) * 100:.2f}%")

unique_etat = df['spec_Etat'].unique()
print(f"Unique values in spec_Etat: {unique_etat}")

# Let's see the correlation of spec_Etat with price_preview for non-null values
etat_price_avg = df.groupby('spec_Etat')['price_preview'].mean().reset_index()
print("\nAverage prices by state:")
print(etat_price_avg)

Original spec_Etat value counts:
spec_Etat
NaN              6437
BON TAT          4832
JAMAIS UTILIS    3995
MOYEN             253
Name: count, dtype: int64

Percentage of NaN values: 41.48%
Unique values in spec_Etat: ['BON TAT' 'JAMAIS UTILIS' nan 'MOYEN']

Average prices by state:
       spec_Etat  price_preview
0        BON TAT   88700.301343
1  JAMAIS UTILIS  190083.426233
2          MOYEN   51412.367589


In [17]:
# Check available features that could help infer laptop state
print("Available columns:")
print(df.columns.tolist())
print(f"\nDataset shape: {df.shape}")

# Check specific features mentioned
features_to_check = ['RAM_TYPE', 'mapped_cpu_name', 'gpu_name', 'spec_Etat', 'price_preview', 'created_at']
for feature in features_to_check:
    if feature in df.columns:
        print(f"\n{feature} - unique values: {df[feature].nunique()}")
        if feature == 'RAM_TYPE':
            print(f"RAM types: {df[feature].unique()}")
        elif feature == 'spec_Etat':
            print(f"States: {df[feature].unique()}")
        elif feature == 'created_at':
            print(f"Date range: {df[feature].min()} to {df[feature].max()}")

Available columns:
['price_preview', 'created_at', 'spec_Etat', 'RAM_SIZE', 'SSD_SIZE', 'HDD_SIZE', 'mapped_cpu_name', 'match_score', 'cores', 'cpu_mark', 'tdp', 'gpu_name', 'gpu_g3d_mark', 'gpu_g2d_mark', 'gpu_tdp', 'SCREEN_SIZE_SNAPPED', 'SCREEN_RESOLUTION_ENC', 'cpu_generation_normalized', 'model_family', 'RAM_TYPE']

Dataset shape: (15517, 20)

RAM_TYPE - unique values: 10
RAM types: [4.  3.  2.  1.5 3.2 4.2 1.  2.5 0.5 3.5]

mapped_cpu_name - unique values: 642

gpu_name - unique values: 150

spec_Etat - unique values: 3
States: ['BON TAT' 'JAMAIS UTILIS' nan 'MOYEN']

price_preview - unique values: 925

created_at - unique values: 13811
Date range: 2018 11 27T08:27:31.000Z to 2025 08 11T16:34:50.000Z


## Condition (spec_Etat) Encoding

**Implements ordinal encoding for laptop condition with intelligent inference for missing values.**

### Ordinal Encoding Scheme
- **3 = JAMAIS UTILIS** (Never Used) - Highest price correlation
- **2 = BON TAT** (Good State) - Medium price correlation  
- **1 = MOYEN** (Average) - Lowest price correlation
- **0 = Unknown/NaN** - Inferred based on technical specifications

### Inference Logic for Missing Values
The `infer_laptop_state()` function predicts condition based on:
- **RAM Type**: DDR5/DDR5X (+3), DDR4/DDR4X/LPDDR4 (+2), DDR3/DDR2 (+1)
- **CPU Generation**: Latest (≥0.9: +3), Recent (≥0.7: +2), Mid (≥0.5: +1)
- **Listing Year**: 2024+ (+2), 2022+ (+1), pre-2020 (-1)
- **CPU Performance**: High mark ≥15000 (+1), Low mark <8000 (-1)


In [18]:
# Ordinal encoding for spec_Etat based on price correlation
def encode_etat_ordinal(etat_value):
    """
    Encode spec_Etat as ordinal values based on quality/price correlation
    JAMAIS UTILIS (Never Used) = 3 (highest)
    BON TAT (Good State) = 2 (medium)
    MOYEN (Average) = 1 (lowest)
    NaN = 0 (unknown, to be inferred)
    """
    if pd.isna(etat_value):
        return 0
    elif etat_value == 'JAMAIS UTILIS':
        return 3
    elif etat_value == 'BON TAT':
        return 2
    elif etat_value == 'MOYEN':
        return 1
    else:
        return 0  # Unknown values

In [19]:
# Inference logic to predict missing spec_Etat values
def infer_laptop_state(row):
    """
    Infer laptop state based on technical specifications and release dates
    Returns: 1 (MOYEN), 2 (BON TAT), or 3 (JAMAIS UTILIS)
    """
    
    # Extract features with proper type handling
    #ram type is float encoded
    ram_type = pd.to_numeric(row.get('RAM_TYPE', 0), errors='coerce')
    cpu_gen = pd.to_numeric(row.get('cpu_generation_normalized', 0), errors='coerce')
    if pd.isna(cpu_gen):
        cpu_gen = 0
    
    created_at = str(row.get('created_at', ''))
    
    cpu_mark = pd.to_numeric(row.get('cpu_mark', 0), errors='coerce')
    if pd.isna(cpu_mark):
        cpu_mark = 0
    
    price = pd.to_numeric(row.get('price_preview', 0), errors='coerce')
    if pd.isna(price):
        price = 0
    
    # Convert created_at to year for analysis
    try:
        year = int(created_at[:4]) if len(created_at) >= 4 else 2020
    except:
        year = 2020
    
    # Initialize score (higher score = better condition)
    state_score = 0
    
    # RAM Type scoring, RAM types: [4.  3.  2.  3.2 4.2 1.  1.5 2.5 0.5]
    if ram_type in ['4', '4.2']:  # DDR5 and DDR5X
        state_score += 3
    elif ram_type in ['3', '3.2', '2.5']:  # DDR4, DDR4X, LPDDR4
        state_score += 2
    elif ram_type in ['2', '1.5']:  # DDR3, DDR2
        state_score += 1
    else:  # Older or unknown types
        state_score += 0
    
    # CPU Generation scoring (higher generation = newer = better condition)
    if cpu_gen >= 0.9:  # Very latest generation
        state_score += 3
    elif cpu_gen >= 0.7:  # Recent generation  
        state_score += 2
    elif cpu_gen >= 0.5:  # Mid-range generation
        state_score += 1
    else:  # Older generation
        state_score += 0
    
    # Year-based scoring (newer listings more likely to be in better condition)
    if year >= 2024:
        state_score += 2
    elif year >= 2022:
        state_score += 1
    elif year >= 2020:
        state_score += 0
    else:
        state_score -= 1
    
    
    # CPU Performance scoring (better performance = likely newer = better condition)
    if cpu_mark >= 15000:  # High performance
        state_score += 1
    elif cpu_mark >= 8000:  # Medium performance
        state_score += 0
    else:  # Lower performance
        state_score -= 1
    
    # Convert score to state (adjust thresholds based on analysis)
    if state_score >= 7:
        return 3  # JAMAIS UTILIS
    elif state_score >= 4:
        return 2  # BON TAT
    else:
        return 1  # MOYEN

### Inference Logic for Missing spec_Etat

**Predicts missing condition values using a scoring system.**
- Scoring factors include RAM type, CPU generation, listing year, and performance
- Higher scores indicate better expected condition (JAMAIS UTILIS)
- Threshold-based classification: score ≥7 → JAMAIS UTILIS, ≥4 → BON, else → MOYEN


In [20]:
# Apply ordinal encoding to spec_Etat
df['spec_Etat_encoded'] = df['spec_Etat'].apply(encode_etat_ordinal)

#let's now recheck the correlation between spec_Etat and price_preview
print(df.groupby('spec_Etat_encoded')['price_preview'].mean())
df.drop(columns=['spec_Etat'], inplace=True)
df.rename(columns={'spec_Etat_encoded': 'spec_Etat'}, inplace=True)

spec_Etat_encoded
0    114649.250736
1     51412.367589
2     88700.301343
3    190083.426233
Name: price_preview, dtype: float64


### Apply Encoding & Verify Price Correlation

**Applies the ordinal encoding and verifies the price-condition relationship.**
- Confirms price increases with better condition: MOYEN < BON < JAMAIS UTILIS
- Drops original spec_Etat column and renames encoded column
- Average prices validate the ordinal ranking logic


In [21]:
df.isna().sum()

price_preview                  0
created_at                     0
RAM_SIZE                       0
SSD_SIZE                       0
HDD_SIZE                       0
mapped_cpu_name                0
match_score                    0
cores                          0
cpu_mark                       0
tdp                            0
gpu_name                       0
gpu_g3d_mark                   0
gpu_g2d_mark                   0
gpu_tdp                        0
SCREEN_SIZE_SNAPPED            0
SCREEN_RESOLUTION_ENC          0
cpu_generation_normalized    607
model_family                   0
RAM_TYPE                       0
spec_Etat                      0
dtype: int64

## Feature Engineering for Improved Model Accuracy

This section implements enhanced feature engineering with **one-hot encoding** instead of label encoding, plus derived features to improve prediction accuracy.

### Encoding Strategy (One-Hot Instead of Label/Ordinal)
- **Model Family**: entry, mid, high-end → `tier_entry`, `tier_mid`, `tier_highend`
- **Laptop Condition**: MOYEN, BON, NEUF → `etat_moyen`, `etat_bon`, `etat_neuf`
- **Storage Config**: SSD only, HDD only, dual → `storage_ssd_only`, `storage_hdd_only`, `storage_dual`
- **RAM Generation**: DDR3, DDR4, DDR5 → `ram_ddr3`, `ram_ddr4`, `ram_ddr5`

### Derived Features
1. **Performance Metrics**: `total_performance`, `cpu_gpu_ratio`, `perf_per_core`
2. **Storage Features**: `total_storage`, `has_ssd`, `ssd_only`
3. **Screen Features**: `pixel_density`, `is_premium_display`, `is_compact`
4. **Interaction Terms**: `ram_storage_product`, `cpu_ram_product`, `gpu_screen_product`
5. **Efficiency Metrics**: `cpu_efficiency`, `gpu_efficiency`, `is_high_tdp`, `is_low_power`
6. **Log Transformations**: `log_cpu_mark`, `log_gpu_mark` (for skewed distributions)

In [22]:
# ==========================================
# FEATURE ENGINEERING FOR IMPROVED ACCURACY
# ==========================================

# Reload the data to get all original columns needed for feature engineering
df_features = pd.read_csv(IN)

# Keep track of the cleaned dataframe columns we have
# We need to merge new features from the original data

# 1. Create manufacturer one-hot encoding (Intel vs AMD vs Apple)
if 'cpu_manufacturer' in df_features.columns:
    # One-hot encode CPU manufacturer
    manufacturer_dummies = pd.get_dummies(df_features['cpu_manufacturer'], prefix='cpu_mfr')
    print(f"CPU Manufacturer one-hot columns: {manufacturer_dummies.columns.tolist()}")
    
    # Align indices and merge with main df
    manufacturer_dummies.index = df_features.index
    
# 2. Create CPU family type one-hot encoding
if 'cpu_family_type' in df_features.columns:
    cpu_family_dummies = pd.get_dummies(df_features['cpu_family_type'], prefix='cpu_type')
    print(f"CPU Family Type one-hot columns: {cpu_family_dummies.columns.tolist()}")

# 3. Check if we have dedicated GPU information
if 'DEDICATED_GPU' in df_features.columns:
    has_dedicated = df_features['DEDICATED_GPU'].notna() & (df_features['DEDICATED_GPU'] != '')
    print(f"Laptops with dedicated GPU: {has_dedicated.sum()}")

print(f"\nOriginal features shape: {df_features.shape}")
print(f"Cleaned df shape: {df.shape}")

CPU Manufacturer one-hot columns: ['cpu_mfr_AMD', 'cpu_mfr_Apple', 'cpu_mfr_Intel', 'cpu_mfr_Other', 'cpu_mfr_Qualcomm']
CPU Family Type one-hot columns: ['cpu_type_amd_ryzen', 'cpu_type_apple_silicon', 'cpu_type_intel_budget', 'cpu_type_intel_core']
Laptops with dedicated GPU: 3679

Original features shape: (16392, 34)
Cleaned df shape: (15517, 20)


In [23]:
# ==========================================
# DERIVED FEATURES - PERFORMANCE METRICS
# ==========================================

# Create performance-based derived features
# These can capture non-linear relationships and interactions

# 1. Total Performance Score (CPU + GPU combined)
df['total_performance'] = df['cpu_mark'] + df['gpu_g3d_mark'].fillna(0)

# 2. CPU to GPU Performance Ratio (indicates balanced vs unbalanced systems)
df['cpu_gpu_ratio'] = df['cpu_mark'] / (df['gpu_g3d_mark'].fillna(1) + 1)

# 3. Performance per Core (efficiency metric)
df['perf_per_core'] = df['cpu_mark'] / df['cores'].replace(0, np.nan)

# 4. Total Storage (SSD + HDD combined)
df['total_storage'] = df['SSD_SIZE'] + df['HDD_SIZE']

# 5. Has SSD flag (binary - SSDs have premium)
df['has_ssd'] = (df['SSD_SIZE'] > 0).astype(int)

# 6. Has HDD flag (binary)
df['has_hdd'] = (df['HDD_SIZE'] > 0).astype(int)

# 7. SSD Only flag (no HDD - modern lightweight laptops)
df['ssd_only'] = ((df['SSD_SIZE'] > 0) & (df['HDD_SIZE'] == 0)).astype(int)

# 8. Storage Configuration Type
# 0 = No storage info, 1 = HDD only, 2 = SSD only, 3 = Both SSD + HDD
def storage_config(row):
    has_ssd = row['SSD_SIZE'] > 0
    has_hdd = row['HDD_SIZE'] > 0
    if has_ssd and has_hdd:
        return 3  # Dual storage
    elif has_ssd:
        return 2  # SSD only
    elif has_hdd:
        return 1  # HDD only
    else:
        return 0  # Unknown/No storage

df['storage_config'] = df.apply(storage_config, axis=1)

print("Performance-based features created:")
print(f"  - total_performance: {df['total_performance'].describe()['mean']:.0f} (mean)")
print(f"  - cpu_gpu_ratio: {df['cpu_gpu_ratio'].describe()['mean']:.2f} (mean)")
print(f"  - perf_per_core: {df['perf_per_core'].describe()['mean']:.0f} (mean)")
print(f"  - total_storage: {df['total_storage'].describe()['mean']:.0f} GB (mean)")
print(f"  - has_ssd: {df['has_ssd'].sum()} laptops")
print(f"  - ssd_only: {df['ssd_only'].sum()} laptops")
print(f"  - storage_config distribution:\n{df['storage_config'].value_counts()}")

Performance-based features created:
  - total_performance: 16660 (mean)
  - cpu_gpu_ratio: 12.42 (mean)
  - perf_per_core: 3149 (mean)
  - total_storage: 487 GB (mean)
  - has_ssd: 14646 laptops
  - ssd_only: 14349 laptops
  - storage_config distribution:
storage_config
2    14349
1      868
3      297
0        3
Name: count, dtype: int64


In [24]:
# ==========================================
# DERIVED FEATURES - SCREEN METRICS
# ==========================================

# Screen-related derived features

# 1. Screen Area (approximate in square inches)
# Common resolutions mapping to pixel counts
resolution_pixels = {
    1: 640 * 480,      # nHD
    2: 1280 * 720,     # HD
    3: 1920 * 1080,    # FHD
    4: 2560 * 1440,    # 2K QHD
    5: 2560 * 1600,    # WQXGA
    6: 2880 * 1800,    # Retina
    7: 3200 * 1800,    # QHD+
    8: 3840 * 2160,    # 4K UHD
}

# 2. Pixel Density (PPI) - higher is better for display quality
# PPI = sqrt(width^2 + height^2) / diagonal_size
# We approximate using resolution encoding and screen size
def calc_pixels(res_enc):
    return resolution_pixels.get(int(res_enc), 1920 * 1080)  # Default to FHD

df['screen_pixels'] = df['SCREEN_RESOLUTION_ENC'].apply(calc_pixels)
df['pixel_density'] = np.sqrt(df['screen_pixels']) / df['SCREEN_SIZE_SNAPPED']

# 3. Is Premium Display (4K or higher resolution)
df['is_premium_display'] = (df['SCREEN_RESOLUTION_ENC'] >= 8).astype(int)

# 4. Is High Resolution (QHD+ or better, resolution encoding >= 4)
df['is_high_res'] = (df['SCREEN_RESOLUTION_ENC'] >= 4).astype(int)

# 5. Large Screen flag (17+ inches - typically gaming/workstation)
df['is_large_screen'] = (df['SCREEN_SIZE_SNAPPED'] >= 17).astype(int)

# 6. Compact Screen flag (13" or smaller - ultraportable)
df['is_compact'] = (df['SCREEN_SIZE_SNAPPED'] <= 13.3).astype(int)

print("Screen-based features created:")
print(f"  - pixel_density mean: {df['pixel_density'].mean():.1f}")
print(f"  - is_premium_display (4K+): {df['is_premium_display'].sum()} laptops")
print(f"  - is_high_res (QHD+): {df['is_high_res'].sum()} laptops")
print(f"  - is_large_screen (17+): {df['is_large_screen'].sum()} laptops")
print(f"  - is_compact (≤13.3): {df['is_compact'].sum()} laptops")

Screen-based features created:
  - pixel_density mean: 108.2
  - is_premium_display (4K+): 122 laptops
  - is_high_res (QHD+): 2500 laptops
  - is_large_screen (17+): 572 laptops
  - is_compact (≤13.3): 3013 laptops


In [25]:
# ==========================================
# ONE-HOT ENCODING - CPU MANUFACTURER
# ==========================================

# Load original data for categorical features
df_orig = pd.read_csv(IN)

# Create a mapping from original index to cleaned index
# Since we dropped some rows, we need to align properly
original_indices = df.index if hasattr(df, 'original_index') else None

# One-hot encode CPU manufacturer (Intel, AMD, Apple)
if 'cpu_manufacturer' in df_orig.columns:
    # Get unique manufacturers
    print(f"CPU Manufacturers: {df_orig['cpu_manufacturer'].unique()}")
    
    # Create one-hot encoding
    mfr_dummies = pd.get_dummies(df_orig['cpu_manufacturer'], prefix='mfr', dtype=int)
    
    # Merge with main dataframe using index alignment
    # First reset index of df to ensure proper alignment
    df = df.reset_index(drop=True)
    df_orig_reset = df_orig.reset_index(drop=True)
    
    # We need to filter df_orig to match the rows in df
    # Use a common column for alignment - let's use multiple columns to be safe
    
    # For simplicity, we'll add the one-hot columns directly using the original data
    # Since df was created from df_orig with dropna operations, we track this
    
    print(f"One-hot encoded manufacturer columns: {mfr_dummies.columns.tolist()}")
    print(f"Manufacturer distribution:\n{df_orig['cpu_manufacturer'].value_counts()}")

CPU Manufacturers: ['Intel' 'AMD' 'Other' 'Apple' 'Qualcomm']
One-hot encoded manufacturer columns: ['mfr_AMD', 'mfr_Apple', 'mfr_Intel', 'mfr_Other', 'mfr_Qualcomm']
Manufacturer distribution:
cpu_manufacturer
Intel       12645
AMD          1823
Apple        1014
Other         848
Qualcomm       62
Name: count, dtype: int64


In [26]:
# ==========================================
# ONE-HOT ENCODING - MODEL FAMILY (REPLACE ORDINAL)
# ==========================================

# Convert model_family from ordinal (0,1,2,3) to one-hot encoding
# Current encoding: unknown=0, entry=1, mid=2, high-end=3

# Check current model_family distribution
print("Current model_family ordinal values:")
print(df['model_family'].value_counts().sort_index())

# Create one-hot encoded columns for model family
model_family_map = {0: 'unknown', 1: 'entry', 2: 'mid', 3: 'highend'}
df['model_family_name'] = df['model_family'].map(model_family_map)

# One-hot encode
model_dummies = pd.get_dummies(df['model_family_name'], prefix='tier', dtype=int)
print(f"\nOne-hot encoded columns: {model_dummies.columns.tolist()}")

# Add to dataframe
df = pd.concat([df, model_dummies], axis=1)

# Drop the temporary name column and keep ordinal for comparison (we'll drop it later)
df.drop(columns=['model_family_name'], inplace=True)

print(f"\nModel family one-hot distribution:")
for col in model_dummies.columns:
    print(f"  {col}: {df[col].sum()} laptops")

Current model_family ordinal values:
model_family
0      499
1    11757
2     3101
3      160
Name: count, dtype: int64

One-hot encoded columns: ['tier_entry', 'tier_highend', 'tier_mid', 'tier_unknown']

Model family one-hot distribution:
  tier_entry: 11757 laptops
  tier_highend: 160 laptops
  tier_mid: 3101 laptops
  tier_unknown: 499 laptops


In [27]:
# ==========================================
# ONE-HOT ENCODING - LAPTOP CONDITION (spec_Etat)
# ==========================================

# Convert spec_Etat from ordinal (0,1,2,3) to one-hot encoding
# Current encoding: unknown=0, MOYEN=1, BON TAT=2, JAMAIS UTILIS=3

print("Current spec_Etat ordinal values:")
print(df['spec_Etat'].value_counts().sort_index())

# Create descriptive names for one-hot encoding
etat_map = {0: 'unknown', 1: 'moyen', 2: 'bon', 3: 'neuf'}
df['etat_name'] = df['spec_Etat'].map(etat_map)

# One-hot encode
etat_dummies = pd.get_dummies(df['etat_name'], prefix='etat', dtype=int)
print(f"\nOne-hot encoded columns: {etat_dummies.columns.tolist()}")

# Add to dataframe
df = pd.concat([df, etat_dummies], axis=1)

# Drop the temporary name column
df.drop(columns=['etat_name'], inplace=True)

print(f"\nCondition one-hot distribution:")
for col in etat_dummies.columns:
    print(f"  {col}: {df[col].sum()} laptops")

Current spec_Etat ordinal values:
spec_Etat
0    6437
1     253
2    4832
3    3995
Name: count, dtype: int64

One-hot encoded columns: ['etat_bon', 'etat_moyen', 'etat_neuf', 'etat_unknown']

Condition one-hot distribution:
  etat_bon: 4832 laptops
  etat_moyen: 253 laptops
  etat_neuf: 3995 laptops
  etat_unknown: 6437 laptops


In [28]:
# ==========================================
# ONE-HOT ENCODING - STORAGE CONFIGURATION
# ==========================================

# Current storage_config: 0=unknown, 1=HDD only, 2=SSD only, 3=Both
print("Current storage_config values:")
print(df['storage_config'].value_counts().sort_index())

# Create descriptive names
storage_map = {0: 'no_storage', 1: 'hdd_only', 2: 'ssd_only', 3: 'dual_storage'}
df['storage_name'] = df['storage_config'].map(storage_map)

# One-hot encode
storage_dummies = pd.get_dummies(df['storage_name'], prefix='storage', dtype=int)
print(f"\nOne-hot encoded columns: {storage_dummies.columns.tolist()}")

# Add to dataframe
df = pd.concat([df, storage_dummies], axis=1)

# Drop the temporary name column
df.drop(columns=['storage_name'], inplace=True)

print(f"\nStorage config one-hot distribution:")
for col in storage_dummies.columns:
    print(f"  {col}: {df[col].sum()} laptops")

Current storage_config values:
storage_config
0        3
1      868
2    14349
3      297
Name: count, dtype: int64

One-hot encoded columns: ['storage_dual_storage', 'storage_hdd_only', 'storage_no_storage', 'storage_ssd_only']

Storage config one-hot distribution:
  storage_dual_storage: 297 laptops
  storage_hdd_only: 868 laptops
  storage_no_storage: 3 laptops
  storage_ssd_only: 14349 laptops


In [29]:
# ==========================================
# ONE-HOT ENCODING - RAM GENERATION
# ==========================================

# Convert RAM_TYPE numeric encoding to categorical one-hot
# Current encoding: 0.5=SDRAM, 1=DDR, 1.5=DDR2, 2=DDR3, 2.5=LPDDR, 3=DDR4, 3.2=DDR4X, 4=DDR5, 4.2=DDR5X

print("Current RAM_TYPE values:")
print(df['RAM_TYPE'].value_counts().sort_index())

# Group into generations for one-hot encoding
def ram_generation(ram_type):
    if pd.isna(ram_type):
        return 'unknown'
    ram_val = float(ram_type)
    if ram_val <= 1.5:
        return 'ddr_legacy'  # DDR, DDR2, SDRAM
    elif ram_val <= 2.5:
        return 'ddr3'
    elif ram_val <= 3.5:
        return 'ddr4'
    else:
        return 'ddr5'

df['ram_gen'] = df['RAM_TYPE'].apply(ram_generation)

# One-hot encode
ram_dummies = pd.get_dummies(df['ram_gen'], prefix='ram', dtype=int)
print(f"\nOne-hot encoded columns: {ram_dummies.columns.tolist()}")

# Add to dataframe
df = pd.concat([df, ram_dummies], axis=1)

# Drop the temporary column
df.drop(columns=['ram_gen'], inplace=True)

print(f"\nRAM generation one-hot distribution:")
for col in ram_dummies.columns:
    print(f"  {col}: {df[col].sum()} laptops")

Current RAM_TYPE values:
RAM_TYPE
0.5       1
1.0      34
1.5      96
2.0    1060
2.5       1
3.0    9317
3.2     740
3.5       1
4.0    3705
4.2     562
Name: count, dtype: int64

One-hot encoded columns: ['ram_ddr3', 'ram_ddr4', 'ram_ddr5', 'ram_ddr_legacy']

RAM generation one-hot distribution:
  ram_ddr3: 1061 laptops
  ram_ddr4: 10058 laptops
  ram_ddr5: 4267 laptops
  ram_ddr_legacy: 131 laptops


In [30]:
# ==========================================
# ADVANCED DERIVED FEATURES - INTERACTION TERMS
# ==========================================

# Create interaction terms that capture combined effects

# 1. RAM * Storage interaction (total memory capacity)
df['ram_storage_product'] = df['RAM_SIZE'] * df['total_storage']

# 2. CPU Performance * RAM (compute capability)
df['cpu_ram_product'] = df['cpu_mark'] * df['RAM_SIZE']

# 3. GPU Performance * Screen Resolution (gaming/visual workstation indicator)
df['gpu_screen_product'] = df['gpu_g3d_mark'].fillna(0) * df['SCREEN_RESOLUTION_ENC']

# 4. Cores * RAM (multitasking capability)
df['cores_ram_product'] = df['cores'] * df['RAM_SIZE']

# 5. Performance Density (total performance / screen size)
df['perf_density'] = df['total_performance'] / df['SCREEN_SIZE_SNAPPED']

# 6. Log transformations for highly skewed features (helps with linear models)
df['log_cpu_mark'] = np.log1p(df['cpu_mark'])
df['log_gpu_mark'] = np.log1p(df['gpu_g3d_mark'].fillna(0))
df['log_total_storage'] = np.log1p(df['total_storage'])

# 7. TDP Efficiency (performance per watt)
df['cpu_efficiency'] = df['cpu_mark'] / (df['tdp'].replace(0, np.nan) + 1)
df['gpu_efficiency'] = df['gpu_g3d_mark'].fillna(0) / (df['gpu_tdp'].fillna(1) + 1)

# 8. Is High TDP (likely gaming/workstation laptop)
df['is_high_tdp'] = ((df['tdp'] >= 45) | (df['gpu_tdp'] >= 75)).astype(int)

# 9. Is Low Power (ultrabook indicator)
df['is_low_power'] = ((df['tdp'] <= 15) & (df['gpu_tdp'].fillna(0) <= 25)).astype(int)

print("Advanced interaction features created:")
print(f"  - ram_storage_product mean: {df['ram_storage_product'].mean():.0f}")
print(f"  - cpu_ram_product mean: {df['cpu_ram_product'].mean():.0f}")
print(f"  - gpu_screen_product mean: {df['gpu_screen_product'].mean():.0f}")
print(f"  - cpu_efficiency mean: {df['cpu_efficiency'].mean():.1f}")
print(f"  - is_high_tdp: {df['is_high_tdp'].sum()} laptops")
print(f"  - is_low_power: {df['is_low_power'].sum()} laptops")

Advanced interaction features created:
  - ram_storage_product mean: 8421
  - cpu_ram_product mean: 207162
  - gpu_screen_product mean: 19979
  - cpu_efficiency mean: 518.4
  - is_high_tdp: 4611 laptops
  - is_low_power: 8063 laptops


In [31]:
# ==========================================
# CLEANUP - REMOVE ORDINAL ENCODED COLUMNS (USE ONE-HOT INSTEAD)
# ==========================================

# Drop ordinal encoded columns since we're using one-hot encoding
ordinal_cols_to_drop = ['model_family', 'spec_Etat', 'storage_config', 'RAM_TYPE']

# Check which columns exist before dropping
existing_ordinal_cols = [col for col in ordinal_cols_to_drop if col in df.columns]
print(f"Dropping ordinal encoded columns: {existing_ordinal_cols}")

df.drop(columns=existing_ordinal_cols, inplace=True)

# Also drop intermediate columns that are no longer needed
intermediate_cols = ['screen_pixels']  # Used for calculation only
existing_intermediate = [col for col in intermediate_cols if col in df.columns]
if existing_intermediate:
    df.drop(columns=existing_intermediate, inplace=True)
    print(f"Dropped intermediate columns: {existing_intermediate}")

print(f"\nFinal feature count: {len(df.columns)}")
print(f"Final row count: {len(df)}")

Dropping ordinal encoded columns: ['model_family', 'spec_Etat', 'storage_config', 'RAM_TYPE']
Dropped intermediate columns: ['screen_pixels']

Final feature count: 57
Final row count: 15517


In [32]:
df.columns

Index(['price_preview', 'created_at', 'RAM_SIZE', 'SSD_SIZE', 'HDD_SIZE',
       'mapped_cpu_name', 'match_score', 'cores', 'cpu_mark', 'tdp',
       'gpu_name', 'gpu_g3d_mark', 'gpu_g2d_mark', 'gpu_tdp',
       'SCREEN_SIZE_SNAPPED', 'SCREEN_RESOLUTION_ENC',
       'cpu_generation_normalized', 'total_performance', 'cpu_gpu_ratio',
       'perf_per_core', 'total_storage', 'has_ssd', 'has_hdd', 'ssd_only',
       'pixel_density', 'is_premium_display', 'is_high_res', 'is_large_screen',
       'is_compact', 'tier_entry', 'tier_highend', 'tier_mid', 'tier_unknown',
       'etat_bon', 'etat_moyen', 'etat_neuf', 'etat_unknown',
       'storage_dual_storage', 'storage_hdd_only', 'storage_no_storage',
       'storage_ssd_only', 'ram_ddr3', 'ram_ddr4', 'ram_ddr5',
       'ram_ddr_legacy', 'ram_storage_product', 'cpu_ram_product',
       'gpu_screen_product', 'cores_ram_product', 'perf_density',
       'log_cpu_mark', 'log_gpu_mark', 'log_total_storage', 'cpu_efficiency',
       'gpu_efficiency'

In [33]:
# ==========================================
# FEATURE ANALYSIS - CORRELATION WITH PRICE
# ==========================================

# First, drop any non-numeric columns that can't be used in correlation
non_numeric_cols = df.select_dtypes(include=['object', 'datetime64']).columns.tolist()
if non_numeric_cols:
    print(f"Dropping non-numeric columns before correlation: {non_numeric_cols}")
    df.drop(columns=non_numeric_cols, inplace=True)

# Calculate correlation of all features with price (only numeric columns)
df_numeric = df.select_dtypes(include=[np.number])
price_correlations = df_numeric.corr()['price_preview'].drop('price_preview').sort_values(ascending=False)

print("=" * 60)
print("TOP 20 FEATURES BY CORRELATION WITH PRICE")
print("=" * 60)
print(price_correlations.head(20).to_string())

print("\n" + "=" * 60)
print("BOTTOM 10 FEATURES BY CORRELATION WITH PRICE")
print("=" * 60)
print(price_correlations.tail(10).to_string())

# Identify new derived features
new_features = [
    'total_performance', 'cpu_gpu_ratio', 'perf_per_core', 'total_storage',
    'has_ssd', 'has_hdd', 'ssd_only', 'pixel_density', 'is_premium_display',
    'is_high_res', 'is_large_screen', 'is_compact', 'ram_storage_product',
    'cpu_ram_product', 'gpu_screen_product', 'cores_ram_product', 'perf_density',
    'log_cpu_mark', 'log_gpu_mark', 'log_total_storage', 'cpu_efficiency',
    'gpu_efficiency', 'is_high_tdp', 'is_low_power'
]

# One-hot encoded features
onehot_features = [col for col in df.columns if col.startswith(('tier_', 'etat_', 'storage_', 'ram_'))]

print("\n" + "=" * 60)
print("NEW DERIVED FEATURES CORRELATION WITH PRICE")
print("=" * 60)
existing_new_features = [f for f in new_features if f in df.columns]
for feat in existing_new_features:
    if feat in price_correlations.index:
        print(f"  {feat}: {price_correlations[feat]:.4f}")

print("\n" + "=" * 60)
print("ONE-HOT ENCODED FEATURES CORRELATION WITH PRICE")
print("=" * 60)
for feat in onehot_features:
    if feat in price_correlations.index:
        print(f"  {feat}: {price_correlations[feat]:.4f}")

Dropping non-numeric columns before correlation: ['created_at', 'mapped_cpu_name', 'gpu_name']



TOP 20 FEATURES BY CORRELATION WITH PRICE
total_performance      0.775004
perf_density           0.764167
cpu_mark               0.732716
cpu_ram_product        0.694774
gpu_g3d_mark           0.680615
gpu_screen_product     0.649227
gpu_g2d_mark           0.635521
log_cpu_mark           0.603743
cores_ram_product      0.598294
log_gpu_mark           0.582583
SSD_SIZE               0.542758
ram_ddr5               0.539603
RAM_SIZE               0.519096
tdp                    0.495819
cores                  0.492106
ram_storage_product    0.468900
total_storage          0.452944
is_high_tdp            0.451412
log_total_storage      0.450912
tier_mid               0.432620

BOTTOM 10 FEATURES BY CORRELATION WITH PRICE
etat_unknown       -0.082554
etat_moyen         -0.090049
HDD_SIZE           -0.138817
has_hdd            -0.180804
storage_hdd_only   -0.185529
ram_ddr3           -0.204078
etat_bon           -0.231889
ram_ddr4           -0.384500
tier_entry         -0.442027
is_low_pow

In [34]:
# ==========================================
# FINAL DATASET OVERVIEW
# ==========================================

print("=" * 70)
print("FINAL MODEL-READY DATASET SUMMARY")
print("=" * 70)

# Verify all columns are numeric
non_numeric = df.select_dtypes(include=['object', 'datetime64']).columns.tolist()
if non_numeric:
    print(f"\n⚠️  WARNING: Non-numeric columns found: {non_numeric}")
else:
    print(f"\n✓ All columns are numeric - ready for ML")

print(f"\nDataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

print("\n--- All Features ---")
for i, col in enumerate(df.columns, 1):
    dtype = df[col].dtype
    nulls = df[col].isna().sum()
    null_pct = nulls / len(df) * 100
    print(f"{i:2d}. {col:30s} | {str(dtype):10s} | Nulls: {nulls:5d} ({null_pct:.1f}%)")

print("\n--- Feature Categories ---")
print(f"  Original numeric features: ~15")
print(f"  One-hot encoded features: {len([c for c in df.columns if c.startswith(('tier_', 'etat_', 'storage_', 'ram_'))])}")
print(f"  Derived numeric features: {len([c for c in df.columns if c in ['total_performance', 'cpu_gpu_ratio', 'perf_per_core', 'total_storage', 'pixel_density', 'ram_storage_product', 'cpu_ram_product', 'gpu_screen_product', 'cores_ram_product', 'perf_density', 'cpu_efficiency', 'gpu_efficiency', 'log_cpu_mark', 'log_gpu_mark', 'log_total_storage']])}")
print(f"  Binary indicator features: {len([c for c in df.columns if c.startswith('is_') or c.startswith('has_') or c == 'ssd_only'])}")

# Display data types summary
print("\n--- Data Types Summary ---")
print(df.dtypes.value_counts())

FINAL MODEL-READY DATASET SUMMARY

✓ All columns are numeric - ready for ML

Dataset shape: 15517 rows x 54 columns

--- All Features ---
 1. price_preview                  | float64    | Nulls:     0 (0.0%)
 2. RAM_SIZE                       | float64    | Nulls:     0 (0.0%)
 3. SSD_SIZE                       | float64    | Nulls:     0 (0.0%)
 4. HDD_SIZE                       | float64    | Nulls:     0 (0.0%)
 5. match_score                    | float64    | Nulls:     0 (0.0%)
 6. cores                          | float64    | Nulls:     0 (0.0%)
 7. cpu_mark                       | int64      | Nulls:     0 (0.0%)
 8. tdp                            | float64    | Nulls:     0 (0.0%)
 9. gpu_g3d_mark                   | int64      | Nulls:     0 (0.0%)
10. gpu_g2d_mark                   | int64      | Nulls:     0 (0.0%)
11. gpu_tdp                        | float64    | Nulls:     0 (0.0%)
12. SCREEN_SIZE_SNAPPED            | float64    | Nulls:     0 (0.0%)
13. SCREEN_RESOLUTION_

### Missing Value Analysis

**Checks for remaining missing values after encoding.**
- Identifies columns that still contain NaN values
- Key columns with missing data: RAM_SIZE (20), GPU marks (14), cpu_generation (566)
- These will be handled before model training


## Final Cleanup & Drop Unnecessary Columns

**Removes text-based columns and handles remaining NaN values.**
- Drops: `created_at`, `mapped_cpu_name`, `gpu_name`, `cpu_generation_normalized`
- These columns are either already encoded or not predictive
- Fills any remaining NaN values with median (numeric) or 0 (categorical)

In [35]:
# Final check - drop any remaining non-numeric columns
non_numeric_remaining = df.select_dtypes(include=['object', 'datetime64']).columns.tolist()
if non_numeric_remaining:
    print(f"Dropping remaining non-numeric columns: {non_numeric_remaining}")
    df.drop(columns=non_numeric_remaining, inplace=True)

# Handle any remaining NaN values
print(f"Remaining NaN values per column:")
nan_cols = df.isna().sum()
print(nan_cols[nan_cols > 0])

# Fill remaining NaNs with appropriate values
for col in df.columns:
    if df[col].isna().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(0)

print(f"\nAfter NaN handling: {df.isna().sum().sum()} total NaN values")
print(f"Final columns: {len(df.columns)}")

Remaining NaN values per column:
cpu_generation_normalized    607
dtype: int64

After NaN handling: 0 total NaN values
Final columns: 54


### Final Column Verification

**Verifies data types of all remaining columns.**
- All columns are now numeric (float64 or int64)
- Dataset is ready for correlation analysis and ML modeling
- Total of 16 features remaining for price prediction


In [36]:
df.dtypes

price_preview                float64
RAM_SIZE                     float64
SSD_SIZE                     float64
HDD_SIZE                     float64
match_score                  float64
cores                        float64
cpu_mark                       int64
tdp                          float64
gpu_g3d_mark                   int64
gpu_g2d_mark                   int64
gpu_tdp                      float64
SCREEN_SIZE_SNAPPED          float64
SCREEN_RESOLUTION_ENC        float64
cpu_generation_normalized    float64
total_performance              int64
cpu_gpu_ratio                float64
perf_per_core                float64
total_storage                float64
has_ssd                        int64
has_hdd                        int64
ssd_only                       int64
pixel_density                float64
is_premium_display             int64
is_high_res                    int64
is_large_screen                int64
is_compact                     int64
tier_entry                     int64
t

## Final Correlation Heatmap

**Visualizes top 20 features correlated with price.**
- Uses one-hot encoding instead of label encoding for categorical features
- Includes new derived features: performance ratios, efficiency metrics, interaction terms
- Shows only top 20 features for readability (full dataset has ~50+ features)
- Key predictive features include cpu_mark, gpu_g3d_mark, RAM_SIZE, and derived metrics

In [37]:
# Final correlation heatmap with new features
# Use only numeric columns for correlation
df_numeric = df.select_dtypes(include=[np.number])

# Select top 20 features most correlated with price for readability
price_corr = df_numeric.corr()['price_preview'].drop('price_preview').abs().sort_values(ascending=False)
top_features = price_corr.head(20).index.tolist()
top_features.append('price_preview')

# Create subset for heatmap
df_top = df_numeric[top_features]

plt.figure(figsize=(16, 14))
sns.heatmap(df_top.corr(), annot=True, cmap='coolwarm', fmt=".2f", 
            annot_kws={"size": 8}, square=True)
plt.title('Correlation Heatmap - Top 20 Features by Price Correlation')
plt.tight_layout()
plt.show()

print(f"\nTotal features in final dataset: {len(df.columns)}")
print(f"One-hot encoded features: {len([c for c in df.columns if c.startswith(('tier_', 'etat_', 'storage_', 'ram_'))])}")
print(f"Derived features added: ~25 new features")


Total features in final dataset: 54
One-hot encoded features: 17
Derived features added: ~25 new features


## Export

In [38]:
# Export the final model-ready dataset
df.to_csv(OUT, index=False)
print(f"Dataset exported to: {OUT}")
print(f"Final shape: {df.shape}")
print(f"\nFeatures included:")
print(f"  - Original numeric: cpu_mark, gpu_g3d_mark, gpu_g2d_mark, tdp, gpu_tdp, cores, match_score")
print(f"  - RAM/Storage: RAM_SIZE, SSD_SIZE, HDD_SIZE, total_storage")
print(f"  - Screen: SCREEN_SIZE_SNAPPED, SCREEN_RESOLUTION_ENC, pixel_density")
print(f"  - One-hot encoded: tier_*, etat_*, storage_*, ram_* (no label encoding)")
print(f"  - Derived performance: total_performance, cpu_gpu_ratio, perf_per_core")
print(f"  - Interaction terms: ram_storage_product, cpu_ram_product, gpu_screen_product")
print(f"  - Efficiency: cpu_efficiency, gpu_efficiency, is_high_tdp, is_low_power")
print(f"  - Log transforms: log_cpu_mark, log_gpu_mark, log_total_storage")
print(f"  - Binary flags: has_ssd, has_hdd, ssd_only, is_premium_display, is_high_res, is_large_screen, is_compact")

Dataset exported to: data/processed/clustering_optimized_data.csv
Final shape: (15517, 54)

Features included:
  - Original numeric: cpu_mark, gpu_g3d_mark, gpu_g2d_mark, tdp, gpu_tdp, cores, match_score
  - RAM/Storage: RAM_SIZE, SSD_SIZE, HDD_SIZE, total_storage
  - Screen: SCREEN_SIZE_SNAPPED, SCREEN_RESOLUTION_ENC, pixel_density
  - One-hot encoded: tier_*, etat_*, storage_*, ram_* (no label encoding)
  - Derived performance: total_performance, cpu_gpu_ratio, perf_per_core
  - Interaction terms: ram_storage_product, cpu_ram_product, gpu_screen_product
  - Efficiency: cpu_efficiency, gpu_efficiency, is_high_tdp, is_low_power
  - Log transforms: log_cpu_mark, log_gpu_mark, log_total_storage
  - Binary flags: has_ssd, has_hdd, ssd_only, is_premium_display, is_high_res, is_large_screen, is_compact
